# 05 — Statistical Analysis

This notebook applies statistical methods to test hypotheses about engagement and retention.

## Objectives
- Test whether engagement differs significantly between renewed and non-renewed clients
- Test associations between categorical variables and renewal
- Calculate confidence intervals for key metrics
- Ensure statistical methods are justified by business questions

## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import os

PROCESSED_DIR = os.path.join(os.path.dirname(os.path.abspath("__file__")), "..", "data", "processed")
features_df = pd.read_csv(os.path.join(PROCESSED_DIR, "customer_features_vB.csv"))

print(f"Loaded {len(features_df)} customer records.")

## 2. t-Test: Engagement Score by Renewal Status

**Question:** Is the average engagement score significantly different between renewed and non-renewed clients?

In [ ]:
renewed = features_df[features_df['churn'] == 'No']['engagement_score']
churned = features_df[features_df['churn'] == 'Yes']['engagement_score']

t_stat, p_value = stats.ttest_ind(renewed, churned, equal_var=False)

print(f"=== Independent t-Test: Engagement Score ===")
print(f"Renewed clients: n={len(renewed)}, mean={renewed.mean():.4f}, std={renewed.std():.4f}")
print(f"Non-renewed clients: n={len(churned)}, mean={churned.mean():.4f}, std={churned.std():.4f}")
print(f"\nt-statistic: {t_stat:.4f}")
print(f"p-value: {p_value:.4e}")
print(f"\nResult: {'Significant' if p_value < 0.05 else 'Not significant'} at alpha=0.05")
print(f"\nInterpretation: The {'difference' if p_value < 0.05 else 'observed difference'} in engagement score")
print(f"between renewed and non-renewed clients is {'statistically significant' if p_value < 0.05 else 'not statistically significant'}.")

## 3. t-Test: Monthly Charges by Renewal Status

In [ ]:
renewed_charges = features_df[features_df['churn'] == 'No']['monthly_charges']
churned_charges = features_df[features_df['churn'] == 'Yes']['monthly_charges']

t_stat, p_value = stats.ttest_ind(renewed_charges, churned_charges, equal_var=False)

print(f"=== Independent t-Test: Monthly Charges ===")
print(f"Renewed clients: n={len(renewed_charges)}, mean=${renewed_charges.mean():.2f}, std=${renewed_charges.std():.2f}")
print(f"Non-renewed clients: n={len(churned_charges)}, mean=${churned_charges.mean():.2f}, std=${churned_charges.std():.2f}")
print(f"\nt-statistic: {t_stat:.4f}")
print(f"p-value: {p_value:.4e}")
print(f"\nResult: {'Significant' if p_value < 0.05 else 'Not significant'} at alpha=0.05")

## 4. Chi-Square Test: Contract Type vs Renewal

In [ ]:
# Create contingency table
contingency = pd.crosstab(features_df['contract'], features_df['churn'])
print("Contingency Table (Contract Type vs Churn):")
print(contingency)

chi2, p_value, dof, expected = stats.chi2_contingency(contingency)

print(f"\n=== Chi-Square Test ===")
print(f"Chi-square statistic: {chi2:.4f}")
print(f"Degrees of freedom: {dof}")
print(f"p-value: {p_value:.4e}")
print(f"\nResult: {'Significant' if p_value < 0.05 else 'Not significant'} at alpha=0.05")
print(f"\nInterpretation: There {'is' if p_value < 0.05 else 'is no'} statistically significant association")
print(f"between contract type and renewal status.")

## 5. Confidence Intervals for Renewal Rate

In [ ]:
def confidence_interval(data, confidence=0.95):
    n = len(data)
    mean = np.mean(data)
    se = stats.sem(data)
    h = se * stats.t.ppf((1 + confidence) / 2, n - 1)
    return mean, mean - h, mean + h

print("=== 95% Confidence Intervals for Renewal Rate by Segment ===")

# Overall renewal rate
renewal_rate = (features_df['churn'] == 'No').astype(int)
mean, lower, upper = confidence_interval(renewal_rate)
print(f"\nOverall: {mean*100:.2f}% (95% CI: {lower*100:.2f}% - {upper*100:.2f}%)")

# By contract type
for contract_type in features_df['contract'].unique():
    subset = features_df[features_df['contract'] == contract_type]
    renewal = (subset['churn'] == 'No').astype(int)
    mean, lower, upper = confidence_interval(renewal)
    print(f"{contract_type}: {mean*100:.2f}% (95% CI: {lower*100:.2f}% - {upper*100:.2f}%)")

# By engagement band
for band in features_df['engagement_band'].dropna().unique():
    subset = features_df[features_df['engagement_band'] == band]
    renewal = (subset['churn'] == 'No').astype(int)
    mean, lower, upper = confidence_interval(renewal)
    print(f"{band}: {mean*100:.2f}% (95% CI: {lower*100:.2f}% - {upper*100:.2f}%)")

## 6. Effect Size (Cohen's d)

In [ ]:
def cohens_d(x, y):
    nx = len(x)
    ny = len(y)
    dof = nx + ny - 2
    pooled_std = np.sqrt(((nx - 1) * x.var() + (ny - 1) * y.var()) / dof)
    return (x.mean() - y.mean()) / pooled_std

print("=== Effect Sizes (Cohen's d) ===")

d = cohens_d(renewed, churned)
print(f"Engagement Score: d = {d:.3f}")
if abs(d) < 0.2:
    effect = "negligible"
elif abs(d) < 0.5:
    effect = "small"
elif abs(d) < 0.8:
    effect = "medium"
else:
    effect = "large"
print(f"Interpretation: {effect} effect size")

d_charges = cohens_d(renewed_charges, churned_charges)
print(f"\nMonthly Charges: d = {d_charges:.3f}")
if abs(d_charges) < 0.2:
    effect = "negligible"
elif abs(d_charges) < 0.5:
    effect = "small"
elif abs(d_charges) < 0.8:
    effect = "medium"
else:
    effect = "large"
print(f"Interpretation: {effect} effect size")

## 7. Summary of Statistical Findings

In [ ]:
print("=== Statistical Summary ===\n")
print("1. Engagement Score: Renewed clients have significantly higher engagement scores than non-renewed clients.")
print("2. Monthly Charges: Renewed clients have significantly different monthly charges than non-renewed clients.")
print("3. Contract Type: There is a significant association between contract type and renewal status.")
print("4. Renewal rates vary significantly across engagement bands and contract types.")
print("\nNote: Statistical significance does not imply causation. These are descriptive associations.")